## Running Abil

In [5]:
#paths:
from pathlib import Path
#handling data:
import pandas as pd
from yaml import load
from yaml import CLoader as Loader
from datetime import datetime
#abil functions:
from abil.tune import ModelTuner as tune
from abil.predict import ModelPredictor as predict
from abil.post import AbilPostProcessor as post


In [6]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "environment.yml").exists():
            return path
    raise FileNotFoundError("Could not find project root containing environment.yml")


PROJECT_ROOT = find_project_root()
conffile = PROJECT_ROOT / "1-phase example" / "regressor.yml"

# Load model configuration
with conffile.open('r') as f:
    model_config = load(f, Loader=Loader)

model_config['root'] = str(PROJECT_ROOT) + '/'
model_config['local_root'] = str(PROJECT_ROOT) + '/'


In [7]:
# Load training data
targets = pd.read_csv(PROJECT_ROOT / model_config['targets'])
d = pd.read_csv(PROJECT_ROOT / model_config['training'])

# Define target
target = targets['Target'][0]

# Define predictors based on YAML
predictors = model_config['predictors']
d = d.dropna(subset=predictors)

# Split training data into X_train and y
y = d[target]
X_train = d[predictors]

print("finished loading data")


finished loading data


In [8]:
#setup model:
m = tune(X_train, y, model_config)

#run model:
m.train(model='rf', log='both')
m.train(model='knn', log='both')
m.train(model='xgb', log='both')


beginning init
length of y: 179
training regressor
{'regressor__estimator__n_estimators': [250], 'regressor__estimator__max_features': [0.5, 1.0], 'regressor__estimator__max_depth': [25, 51], 'regressor__estimator__min_samples_leaf': [1, 5], 'regressor__estimator__max_samples': [0.9, 1.0]}
Fitting 10 folds for each of 16 candidates, totalling 160 fits
Fitting 10 folds for each of 16 candidates, totalling 160 fits
best fit: log
exported model to: /home/joost/Abil_tutorial/ModelOutput/pp_na/model/rf/Primary_Production_reg.sav


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   2 out of  10 | elapsed:    0.5s remaining:    1.9s


exported scoring to:  /home/joost/Abil_tutorial/ModelOutput/pp_na/scoring/rf/Primary_Production_reg.sav
reg rRMSE: 76%
reg rMAE: 44%
reg R2: 0.46
execution time: 0.0 seconds
training regressor
{'regressor__estimator__max_samples': [0.5, 1.0], 'regressor__estimator__max_features': [2, 4], 'regressor__estimator__estimator__n_neighbors': [3, 4], 'regressor__estimator__estimator__p': [1, 2], 'regressor__estimator__estimator__weights': ['distance']}


[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    0.7s finished


Fitting 10 folds for each of 16 candidates, totalling 160 fits
Fitting 10 folds for each of 16 candidates, totalling 160 fits
best fit: log
exported model to: /home/joost/Abil_tutorial/ModelOutput/pp_na/model/knn/Primary_Production_reg.sav
exported scoring to:  /home/joost/Abil_tutorial/ModelOutput/pp_na/scoring/knn/Primary_Production_reg.sav
reg rRMSE: 81%
reg rMAE: 49%
reg R2: 0.41
execution time: 0.0 seconds
training regressor
{'regressor__estimator__n_estimators': [100], 'regressor__estimator__learning_rate': [0.1, 0.2], 'regressor__estimator__max_depth': [25, 51], 'regressor__estimator__subsample': [0.5, 1.0], 'regressor__estimator__colsample_bytree': [0.9, 1.0], 'regressor__estimator__reg_alpha': [0, 0.1], 'regressor__estimator__gamma': [0]}
Fitting 10 folds for each of 32 candidates, totalling 320 fits


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   2 out of  10 | elapsed:    0.1s remaining:    0.5s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    0.2s finished


Fitting 10 folds for each of 32 candidates, totalling 320 fits
best fit: log
exported model to: /home/joost/Abil_tutorial/ModelOutput/pp_na/model/xgb/Primary_Production_reg.sav
exported scoring to:  /home/joost/Abil_tutorial/ModelOutput/pp_na/scoring/xgb/Primary_Production_reg.sav
reg rRMSE: 73%
reg rMAE: 43%
reg R2: 0.40
execution time: 0.0 seconds


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   2 out of  10 | elapsed:    0.1s remaining:    0.2s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    0.1s finished


In [9]:
# Load prediction data
X_predict = pd.read_csv(PROJECT_ROOT / model_config['prediction'])
X_predict.set_index(["time", "depth", "lat", "lon"], inplace=True)
X_predict = X_predict[predictors]

# Setup model
m = predict(X_train, y, X_predict, model_config)

# Predict model
m.make_prediction()


initialized prediction
number of models in ensemble: 3
predicting regressor
finished exporting summary stats to: /home/joost/Abil_tutorial/ModelOutput/pp_na/predictions/rf/Primary_Production.nc
predicting regressor
finished exporting summary stats to: /home/joost/Abil_tutorial/ModelOutput/pp_na/predictions/xgb/Primary_Production.nc
predicting regressor
finished exporting summary stats to: /home/joost/Abil_tutorial/ModelOutput/pp_na/predictions/knn/Primary_Production.nc
finished exporting summary stats to: /home/joost/Abil_tutorial/ModelOutput/pp_na/predictions/ens/Primary_Production.nc
finished
execution time: 108.1186592578888 seconds


[Parallel(n_jobs=1)]: Done  10 out of  10 | elapsed:    3.4s finished


In [10]:
target_names = targets['Target'].values
target_subset = target_names[:1] # subset for estimate_applicability and merge_obs
current_date = datetime.today().strftime('%Y-%m-%d')

def do_post(statistic):
    m = post(X_train, y, X_predict, model_config, statistic)
    
    if statistic == "mean":
        print('begin estimate_applicability')
        m.estimate_applicability(target_subset)

        print('begin merge_obs')
        m.merge_obs(current_date,target_subset)

    print('begin export_ds')
    m.export_ds(current_date)

    print('begin integration')
    magnitude_conversion = 1e-21
    molar_mass = 12.01
    integ = m.integration(m, magnitude_conversion=magnitude_conversion,molar_mass=molar_mass,rate=True)
    if statistic == "mean":
        integ.integrated_totals(target_names)
    else:
        integ.integrated_totals(target_subset)

    print('do_post for: ', statistic, ' complete')

In [11]:
do_post(statistic="mean")
do_post(statistic="ci95_UL")
do_post(statistic="ci95_LL")

merging... /home/joost/Abil_tutorial/ModelOutput/pp_na/predictions/ens
finished merging NetCDF files
Model configuration exported to: /home/joost/Abil_tutorial/ModelOutput/pp_na/posts/model_config.yml
the target is: Primary_Production
finished merging parameters
the target is: Primary_Production
finished merging parameters
the target is: Primary_Production
finished merging parameters
models included in merge performance! ['rf', 'xgb', 'knn']
finished merging performance
finished merging performance
finished merging performance
finished merging performance
begin estimate_applicability
begin merge_obs
                       Primary_Production  Primary_Production_mod  \
lat  lon   depth time                                               
46.0 -8.0  0.0   6                1535.14             1576.687955   
           5.0   6                1620.71             1546.151625   
           15.0  6                1548.27             1391.839454   
           20.0  6                1236.99       